Prepared only; not launched. Internet off. Attach GPT2-medium revision 6dcaa7a952f72f9298047fd5137cd6e4f05f41da including model/tokenizer files and revision-manifest.json containing model, revision, and relative file SHA256 mapping. Requires >=2GB model storage and approximately 4GB working RAM (estimate). Root must verify attachment paths and quota before execution.

In [ ]:
from pathlib import Path
import json, subprocess, sys
Path('/kaggle/working/rank_transitions.py').write_text('"""Frozen short-text TRACE adaptation. No generation, API calls, or fitting in extraction."""\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef summaries(ranks, entropy, surprisal):\n    ranks, entropy, surprisal = map(np.asarray, (ranks, entropy, surprisal))\n    assert ranks.ndim == 1 and len(ranks) == len(entropy) == len(surprisal)\n    assert len(ranks) and np.isfinite(ranks).all() and (ranks >= 1).all()\n    assert np.isfinite(entropy).all() and np.isfinite(surprisal).all()\n    buckets = np.searchsorted([10, 100, 1000], ranks, side=\'left\')\n    marginal = []\n    for ix in np.array_split(np.arange(len(ranks)), 3):\n        if len(ix):\n            marginal.extend(np.bincount(buckets[ix], minlength=4) / len(ix))\n            marginal.extend([entropy[ix].mean(), surprisal[ix].mean(), np.log(ranks[ix]).mean(), surprisal[ix].std()])\n        else:\n            marginal.extend([0.] * 8)\n    pairs = 4*buckets[:-1] + buckets[1:]\n    transitions = np.bincount(pairs, minlength=16) / max(1, len(pairs))\n    return np.r_[marginal, transitions].astype(np.float32)\n\n\ndef shifted_scores(logits, ids, mask):\n    """logits[t] predicts ids[t+1]. Input tensors [B,T,V], [B,T], [B,T]."""\n    import torch\n    logits = logits[:, :-1].float()\n    target = ids[:, 1:]\n    valid = mask[:, 1:].bool() & mask[:, :-1].bool()\n    chosen = logits.gather(-1, target.unsqueeze(-1)).squeeze(-1)\n    rank = 1 + (logits > chosen.unsqueeze(-1)).sum(-1)\n    logp = torch.log_softmax(logits, dim=-1)\n    entropy = -(logp.exp() * logp).sum(-1)\n    surprisal = -logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)\n    return [(rank[i][valid[i]].cpu().numpy(), entropy[i][valid[i]].cpu().numpy(),\n             surprisal[i][valid[i]].cpu().numpy()) for i in range(len(ids))]\n\n\ndef extract(args, plan):\n    import pandas as pd\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    frame = pd.read_csv(args.input, dtype={\'id\': str})\n    assert frame.id.is_unique and frame.text.notna().all()\n    if args.limit: frame = frame.iloc[:args.limit].copy()\n    out = Path(args.output); out.mkdir(parents=True, exist_ok=False)\n    (out/\'protocol.json\').write_bytes(Path(args.plan).read_bytes())\n    start = time.monotonic()\n    source = args.model_path or plan[\'model\']\n    if args.model_path:\n        manifest = json.loads((Path(args.model_path)/\'revision-manifest.json\').read_text())\n        assert manifest[\'model\'] == plan[\'model\'] and manifest[\'revision\'] == plan[\'revision\']\n        assert manifest[\'files\'], \'Model file checksums required\'\n        for relative, digest in manifest[\'files\'].items():\n            assert sha(Path(args.model_path)/relative) == digest, relative\n    options = {\'local_files_only\': True} if args.model_path else {\'revision\': plan[\'revision\']}\n    tokenizer = AutoTokenizer.from_pretrained(source, **options)\n    tokenizer.pad_token = tokenizer.eos_token; tokenizer.padding_side = \'right\'\n    model = AutoModelForCausalLM.from_pretrained(source, torch_dtype=torch.float32,\n                                                attn_implementation=\'eager\', **options).eval().to(args.device)\n    lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in frame.text]\n    assert min(lengths) >= 2\n    # Fail closed, rather than silently discard long response content.\n    assert max(lengths) <= plan[\'max_tokens\'], (\'Long row requires new protocol\', max(lengths))\n    feats, counts, audit = [], [], []\n    with torch.inference_mode():\n        for begin in range(0, len(frame), plan[\'batch_size\']):\n            batch = tokenizer(frame.text.iloc[begin:begin+plan[\'batch_size\']].tolist(),\n                              add_special_tokens=False, padding=True, return_tensors=\'pt\').to(args.device)\n            logits = model(**batch, use_cache=False).logits\n            scores = shifted_scores(logits, batch[\'input_ids\'], batch[\'attention_mask\'])\n            if begin == 0:\n                # Explicit causal/padding parity: standalone prefix vs full padded sequence.\n                for row in range(len(batch[\'input_ids\'])):\n                    n = int(batch[\'attention_mask\'][row].sum())\n                    for pos in sorted(set([0, min(3, n-2), n-2])):\n                        prefix = batch[\'input_ids\'][row:row+1, :pos+1]\n                        serial = model(input_ids=prefix, use_cache=False).logits[0, -1].float()\n                        vector = logits[row, pos].float()\n                        error = float((serial-vector).abs().max().cpu())\n                        assert torch.allclose(serial, vector, atol=3e-4, rtol=3e-4), error\n                        audit.append({\'row\': row, \'position\': pos, \'max_logit_error\': error})\n            for rank, entropy, surprise in scores:\n                feats.append(summaries(rank, entropy, surprise)); counts.append(len(rank))\n            elapsed = time.monotonic()-start\n            progress = {\'rows\': len(feats), \'total\': len(frame), \'seconds\': elapsed}\n            with (out/\'progress.jsonl\').open(\'a\') as f: f.write(json.dumps(progress)+\'\\n\')\n            print(json.dumps(progress), flush=True)\n            if elapsed > plan[\'extraction_budget_seconds\']:\n                raise TimeoutError(\'Frozen extraction budget reached; preserve log and do not extend silently\')\n    np.savez_compressed(out/\'features.npz\', ids=frame.id.to_numpy(dtype=str), features=np.stack(feats), scored_tokens=counts)\n    (out/\'manifest.json\').write_text(json.dumps({\'input_sha256\':sha(args.input), \'protocol_sha256\':sha(out/\'protocol.json\'),\n        \'features_sha256\':sha(out/\'features.npz\'), \'script_sha256\':sha(__file__), \'model\':plan[\'model\'],\n        \'revision\':plan[\'revision\'], \'local_model_path\':args.model_path, \'alignment_checks\':audit,\n        \'rows\':len(frame), \'seconds\':time.monotonic()-start, \'device\':args.device,\n        \'torch\':torch.__version__, \'numpy\':np.__version__}, indent=2))\n\n\ndef evaluate(args, plan):\n    import pandas as pd\n    from scipy.special import softmax\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.metrics import balanced_accuracy_score\n    from threadpoolctl import threadpool_limits\n    host = pd.read_csv(args.input, dtype={\'id\':str, \'prompt_id\':str})\n    folds = pd.read_csv(plan[\'folds\'], dtype={\'id\':str, \'prompt_id\':str})\n    manifest=json.loads((Path(args.features).parent/\'manifest.json\').read_text())\n    assert manifest[\'input_sha256\']==sha(args.input) and manifest[\'features_sha256\']==sha(args.features)\n    assert manifest[\'protocol_sha256\']==sha(args.plan)\n    feat = np.load(args.features, allow_pickle=False)\n    ref = np.load(plan[\'reference\'], allow_pickle=False)\n    assert np.array_equal(host.id, folds.id) and np.array_equal(host.prompt_id, folds.prompt_id)\n    assert np.array_equal(host.id, feat[\'ids\']) and np.array_equal(host.id, ref[\'ids\'])\n    assert np.array_equal(folds.fold, ref[\'fold\']) and host.id.is_unique\n    assert host.assign(fold=folds.fold).groupby(\'prompt_id\').fold.nunique().max() == 1\n    labels = ref[\'labels\']; assert list(labels)==sorted(host.model.unique())\n    assert sorted(folds.fold.unique())==[0,1,2,3,4]\n    y = np.searchsorted(labels, host.model)\n    assert np.array_equal(labels[y], host.model)\n    x = feat[\'features\']; assert x.shape == (len(host),40) and np.isfinite(x).all()\n    assert np.isfinite(ref[plan[\'reference_score_key\']]).all()\n    base = softmax(ref[plan[\'reference_score_key\']], axis=1)\n    allp, records = {}, []\n    with threadpool_limits(limits=2):\n        for name, dim in [(\'marginal\',24),(\'transitions\',40)]:\n            p = np.full_like(base, np.nan)\n            for fold in sorted(folds.fold.unique()):\n                tr = folds.fold.to_numpy()!=fold; va = ~tr\n                assert set(host.prompt_id[tr]).isdisjoint(host.prompt_id[va])\n                fit = make_pipeline(StandardScaler(), LogisticRegression(**plan[\'classifier\']))\n                fit.fit(x[tr,:dim], y[tr]); assert np.array_equal(fit[-1].classes_,np.arange(18))\n                p[va] = fit.predict_proba(x[va,:dim])\n                blend = (1-plan[\'blend_weight\'])*base[va]+plan[\'blend_weight\']*p[va]\n                score = lambda v: float(balanced_accuracy_score(y[va],v.argmax(1)))\n                records.append(dict(arm=name,fold=int(fold),reference=score(base[va]),\n                                    standalone=score(p[va]),blend=score(blend),gain=score(blend)-score(base[va])))\n            allp[name]=p\n    rng=np.random.default_rng(5105); prompts=host.prompt_id.unique(); boots=[]\n    pred=(.9*base+.1*allp[\'transitions\']).argmax(1); bp=base.argmax(1)\n    # Prompt-cluster resampling retains correlated responses from each prompt.\n    groups=[np.flatnonzero(host.prompt_id.to_numpy()==p) for p in prompts]\n    for _ in range(plan[\'bootstrap_replicates\']):\n        ix=np.concatenate([groups[i] for i in rng.integers(len(groups),size=len(groups))])\n        boots.append(balanced_accuracy_score(y[ix],pred[ix])-balanced_accuracy_score(y[ix],bp[ix]))\n    metric=lambda z:float(balanced_accuracy_score(y,z.argmax(1)))\n    gain=metric(.9*base+.1*allp[\'transitions\'])-metric(base)\n    marginal=metric(.9*base+.1*allp[\'marginal\'])\n    ci=np.quantile(boots,[.025,.975]).tolist()\n    result=dict(experiment=plan[\'experiment\'], folds=records, gain=gain, prompt_bootstrap95=ci,\n                passed=bool(gain>=.005 and ci[0]>0 and all(r[\'gain\']>0 for r in records if r[\'arm\']==\'transitions\')\n                            and metric(.9*base+.1*allp[\'transitions\'])>marginal),\n                scope=\'Previously inspected prompt-disjoint component OOF; not full ensemble or LB.\',\n                input_sha256={p:sha(p) for p in [args.input,args.features,args.plan,plan[\'folds\'],plan[\'reference\']]})\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=False)\n    (out/\'result.json\').write_text(json.dumps(result,indent=2))\n    np.savez_compressed(out/\'oof.npz\',ids=host.id.to_numpy(dtype=str),labels=labels,**allp)\n    print(json.dumps(result,indent=2))\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=[\'extract\',\'evaluate\'])\n    p.add_argument(\'--plan\',default=\'configs/name-that-llm/nov5-rank-transitions.json\')\n    p.add_argument(\'--input\',default=\'data/name-that-llm/train.csv\');p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--device\',default=\'cpu\');p.add_argument(\'--model-path\');p.add_argument(\'--limit\',type=int)\n    p.add_argument(\'--features\');a=p.parse_args();plan=json.loads(Path(a.plan).read_text())\n    (extract if a.mode==\'extract\' else evaluate)(a,plan)\n')
Path('/kaggle/working/protocol.json').write_text('{\n  "experiment": "nov5-rank-transitions",\n  "hypothesis": "Consecutive frozen-evaluator token ranks contribute authorship information beyond their marginal distribution and improve a fixed held-out discriminative reference.",\n  "model": "openai-community/gpt2-medium",\n  "revision": "6dcaa7a952f72f9298047fd5137cd6e4f05f41da",\n  "revision_source": "https://huggingface.co/api/models/openai-community/gpt2-medium",\n  "paper": "https://arxiv.org/html/2603.28054v1",\n  "code_reference": "https://github.com/anudeex/TRACE/blob/main/src/generate-token-scores-TRACE.py",\n  "method_scope": "Independent short-paragraph adaptation; paper evidence concerns long books, not this dataset. Frozen evaluator is not a proxy for source tokenizers.",\n  "max_tokens": 1024,\n  "batch_size": 2,\n  "rank_bucket_upper_bounds": [\n    10,\n    100,\n    1000\n  ],\n  "rank_definition": "One plus number of strictly larger logits; ties share rank. No BOS/EOS added; token zero has no score.",\n  "features": "24 marginal/position features plus 16 transition frequencies = 40 dimensions. Each third: four bucket frequencies, mean entropy, mean surprisal, mean log rank, surprisal standard deviation. Empty thirds zero.",\n  "classifier": {\n    "C": 0.1,\n    "max_iter": 2000,\n    "random_state": 5105\n  },\n  "folds": "artifacts/baseline-v1/folds.csv",\n  "reference": "artifacts/improvement-v34/primary-oof.npz",\n  "reference_score_key": "candidate_scores",\n  "blend_weight": 0.1,\n  "gate": "All five fold blend gains positive, pooled balanced-accuracy gain >=0.005, prompt-bootstrap 95% lower bound >0; transitions must outperform marginal-only fixed blend. Previously inspected folds, not pristine confirmation. No full-ensemble leaderboard claim.",\n  "bootstrap_replicates": 2000,\n  "extraction_budget_seconds": 3600,\n  "execution": "Not launched. Start with 100-row throughput/causal-alignment probe. CPU/MPS preferred if benchmark projects <=1 hour. Otherwise root chooses Kaggle execution; no paid compute authorized by this protocol.",\n  "reference_description": "Raw V34 8B component scores, not fixed sparse blend and not full leaderboard ensemble."\n}\n')
# Attach competition and the pinned GPT2-medium model private dataset first.
MODEL_PATH='/kaggle/input/ntl-gpt2-medium-pinned'
TRAIN_PATH='/kaggle/input/name-that-llm/train.csv'
subprocess.run([sys.executable,'/kaggle/working/rank_transitions.py','extract','--plan','/kaggle/working/protocol.json','--input',TRAIN_PATH,'--model-path',MODEL_PATH,'--device','cuda','--output','/kaggle/working/trace-features'],check=True)
